In [9]:
import asyncio
import inspect
import json
from collections.abc import Callable, Mapping, Sequence
from typing import Any

from pydantic import BaseModel, ConfigDict, Field, ValidationError

### Task 1

In [2]:
class ToolSpec(BaseModel):
    model_config = ConfigDict(arbitrary_types_allowed=True)

    name: str
    description: str
    argument_model: type[BaseModel]
    handler: Callable[..., Any] | None = Field(default=None, exclude=True)

    @property
    def parameters(self) -> dict[str, Any]:
        return self.argument_model.model_json_schema()

    def to_openai_tool(self) -> dict[str, Any]:
        return {
            "type": "function",
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
            "strict": False,
        }

    def to_anthropic_tool(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "input_schema": self.parameters,
        }

    def to_google_function_declaration(self) -> dict[str, Any]:
        return {
            "name": self.name,
            "description": self.description,
            "parameters": self.parameters,
        }

### Task 2

In [3]:
def detect_schema_drift(reference: ToolSpec, candidate: ToolSpec) -> dict[str, Any]:
    """Report argument fields added, removed, or retyped between tool definitions."""
    reference_fields = reference.argument_model.model_fields
    candidate_fields = candidate.argument_model.model_fields
    reference_names = set(reference_fields)
    candidate_names = set(candidate_fields)

    type_changes = {}
    for field_name in sorted(reference_names & candidate_names):
        reference_type = reference_fields[field_name].annotation
        candidate_type = candidate_fields[field_name].annotation
        if reference_type != candidate_type:
            type_changes[field_name] = {
                "reference": getattr(reference_type, "__name__", str(reference_type)),
                "candidate": getattr(candidate_type, "__name__", str(candidate_type)),
            }

    added_fields = sorted(candidate_names - reference_names)
    removed_fields = sorted(reference_names - candidate_names)
    return {
        "has_drift": bool(added_fields or removed_fields or type_changes),
        "added_fields": added_fields,
        "removed_fields": removed_fields,
        "type_changes": type_changes,
    }

### Task 3

In [ ]:
async def execute_tool_calls(
    calls: Sequence[Mapping[str, Any]], tools: Sequence[ToolSpec]
) -> list[dict[str, Any]]:
    tools_by_name = {tool.name: tool for tool in tools}

    async def execute(call: Mapping[str, Any]) -> dict[str, Any]:
        result = {"call_id": call.get("id"), "tool_name": call.get("name")}
        try:
            tool = tools_by_name[call["name"]]
            arguments = tool.argument_model.model_validate(call["arguments"])
            value = tool.handler(arguments)
            if inspect.isawaitable(value):
                value = await value
            return {**result, "result": value, "error": None}
        except (ValueError, KeyError) as error:
            return {**result, "result": None, "error": str(error)}

    outcomes = await asyncio.gather(*(execute(call) for call in calls), return_exceptions=True)
    results = []
    for call, outcome in zip(calls, outcomes):
        if isinstance(outcome, BaseException):
            if not isinstance(outcome, Exception):
                raise outcome
            outcome = {"result": None, "error": str(outcome)}
        results.append(
            outcome
            if "call_id" in outcome
            else {"call_id": call.get("id"), "tool_name": call.get("name"), **outcome}
        )
    return results

### Task 4

implemented in test_exercise.py

### Task 5

In [15]:
def compare_malformed_json(
    prompt_outputs: Sequence[str],
    forced_tool_outputs: Sequence[str],
    output_model: type[BaseModel],
) -> dict[str, float | None]:
    def malformed_rate(outputs: Sequence[str]) -> float | None:
        if not outputs:
            return None
        malformed = 0
        for output in outputs:
            try:
                output_model.model_validate(json.loads(output))
            except (json.JSONDecodeError, ValidationError, TypeError):
                malformed += 1
        return malformed / len(outputs)

    return {
        "prompt_json_rate": malformed_rate(prompt_outputs),
        "forced_tool_rate": malformed_rate(forced_tool_outputs),
    }

In [22]:
class SampleOutput(BaseModel):
    value: int


rates = compare_malformed_json(
    ['{"value": 1}', "not json", '{"value": "invalid"}'],
    ['{"value": 2}', '{"value": 3}', '{"value" : "2"}'],
    SampleOutput,
)
print(rates)

{'prompt_json_rate': 0.6666666666666666, 'forced_tool_rate': 0.0}


### Task 6

In [29]:
def validate_schema_for_provider(
    model: type[BaseModel], provider: str, rules_by_provider: Mapping[str, Mapping[str, Any]]
) -> list[str]:
    if provider not in rules_by_provider:
        raise ValueError(f"No rules configured for '{provider}'.")

    rules = rules_by_provider[provider]
    unsupported = set(rules.get("unsupported_keywords", ()))
    max_depth = rules.get("max_depth")
    max_properties = rules.get("max_properties")
    issues = []
    property_count = 0

    def visit(node: Any, path: str = "$", depth: int = 0) -> None:
        nonlocal property_count
        if not isinstance(node, Mapping):
            return
        if max_depth is not None and depth > max_depth:
            issues.append(f"Schema too deep at {path} (limit {max_depth}).")
        issues.extend(
            f"Unsupported '{key}' at {path}." for key in sorted(unsupported & node.keys())
        )
        properties = node.get("properties", {})
        if isinstance(properties, Mapping):
            property_count += len(properties)
        for key, value in node.items():
            if isinstance(value, Mapping):
                children = (
                    value.items()
                    if key in {"properties", "$defs", "definitions"}
                    else [(key, value)]
                )
                for name, child in children:
                    visit(child, f"{path}/{key}/{name}", depth + 1)
            elif isinstance(value, list):
                for index, child in enumerate(value):
                    visit(child, f"{path}/{key}/{index}", depth + 1)

    visit(model.model_json_schema())
    if max_properties is not None and property_count > max_properties:
        issues.append(f"Too many properties: {property_count} (limit {max_properties}).")
    return issues

In [32]:
class ExampleSchema(BaseModel):
    value: int | str
    label: str


provider_rules = {
    "example_provider": {
        "unsupported_keywords": {"anyOf"},
        "max_depth": 3,
        "max_properties": 3,
    }
}
issues = validate_schema_for_provider(ExampleSchema, "example_provider", provider_rules)
print(issues)

["Unsupported 'anyOf' at $/properties/value."]


### Task 7

implemented in test_exercise.py

### Task 8

In [35]:
def log_tool_schemas(tool: ToolSpec) -> dict[str, dict[str, Any]]:
    schemas = {
        "openai": tool.to_openai_tool(),
        "anthropic": tool.to_anthropic_tool(),
        "google": tool.to_google_function_declaration(),
    }
    for provider, schema in schemas.items():
        print(f"[{provider}] {tool.name}\n{json.dumps(schema, indent=2, sort_keys=True)}")
    return schemas

In [37]:
class CallArguments(BaseModel):
    value: int


def handler():
    pass


spec = ToolSpec(
    name="calculate",
    description="Calculate a value.",
    argument_model=CallArguments,
    handler=handler,
)

log_tool_schemas(spec)

[openai] calculate
{
  "description": "Calculate a value.",
  "name": "calculate",
  "parameters": {
    "properties": {
      "value": {
        "title": "Value",
        "type": "integer"
      }
    },
    "required": [
      "value"
    ],
    "title": "CallArguments",
    "type": "object"
  },
  "strict": false,
  "type": "function"
}
[anthropic] calculate
{
  "description": "Calculate a value.",
  "input_schema": {
    "properties": {
      "value": {
        "title": "Value",
        "type": "integer"
      }
    },
    "required": [
      "value"
    ],
    "title": "CallArguments",
    "type": "object"
  },
  "name": "calculate"
}
[google] calculate
{
  "description": "Calculate a value.",
  "name": "calculate",
  "parameters": {
    "properties": {
      "value": {
        "title": "Value",
        "type": "integer"
      }
    },
    "required": [
      "value"
    ],
    "title": "CallArguments",
    "type": "object"
  }
}


{'openai': {'type': 'function',
  'name': 'calculate',
  'description': 'Calculate a value.',
  'parameters': {'properties': {'value': {'title': 'Value',
     'type': 'integer'}},
   'required': ['value'],
   'title': 'CallArguments',
   'type': 'object'},
  'strict': False},
 'anthropic': {'name': 'calculate',
  'description': 'Calculate a value.',
  'input_schema': {'properties': {'value': {'title': 'Value',
     'type': 'integer'}},
   'required': ['value'],
   'title': 'CallArguments',
   'type': 'object'}},
 'google': {'name': 'calculate',
  'description': 'Calculate a value.',
  'parameters': {'properties': {'value': {'title': 'Value',
     'type': 'integer'}},
   'required': ['value'],
   'title': 'CallArguments',
   'type': 'object'}}}

### Task 9

implemented in test_exercise.py

### Task 10

implemented in mini-project tests